# Segmentation Pipeline — Multi-Scale Anomaly Localization

**Thesis Section: Chapter 5**

This notebook runs the full segmentation pipeline:
1. Start persistent Chronos-T5-small subprocess server
2. Extract sliding window embeddings (512-dim each)
3. Encode natural-language queries using ChatTS-14B embed_tokens (5120-dim)
4. MLP (V2, threshold=0.65) predicts per-window anomaly probability
5. Multi-scale aggregation [32, 64, 128, 256] with max-pooling
6. Plot heatmap and binary mask

**Final model**: F1=0.699, Acc=88.0%, Precision=0.752, Recall=0.653

**Requires**: ChatTS-14B loaded, Chronos subprocess environment (chatts_chronos)

In [ ]:
import sys
sys.path.insert(0, '../..')

from dotenv import load_dotenv
load_dotenv('../../.env')

import os
import numpy as np
import torch

from src.data.timeseer_client import fetch_series_api
from src.data.ground_truth import GROUND_TRUTH
from src.models.chatts_loader import load_model
from src.models.mlp import load_mlp
from src.inference.embeddings import encode_text_query
from src.inference.chronos_server import start_server, get_chronos_embedding_cached, shutdown_server
from src.preprocessing.chunking import downsample
from src.segmentation.pipeline import multiscale_segmentation
from src.visualization.segmentation_plots import plot_segmentation, plot_segmentation_heatmap

VSC_SCRATCH = os.environ.get('VSC_SCRATCH', '/scratch/leuven/375/vsc37531')
MLP_PATH    = os.path.join(VSC_SCRATCH, 'ChatTS', 'timeseer_data', 'segmentation_mlp_final.pt')

print('Imports OK')

In [ ]:
# Load ChatTS-14B for text embeddings
model, tokenizer, processor = load_model('ChatTS-14B')

def encode_text(q):
    return encode_text_query(q, model=model, tokenizer=tokenizer)

In [ ]:
# Start Chronos persistent server (separate conda env)
start_server()

# Load trained MLP
mlp = load_mlp(MLP_PATH)
FINAL_THRESHOLD = 0.65

In [ ]:
# Test: R1-AT-102-COND (ground truth: B) Spikes)
TAG  = 'R1-AT-102-COND'
AREA = 'Reactor 1'

vals, idx = fetch_series_api(TAG, AREA)
vals_ds   = downsample(vals, target=1024)

# True anomaly at idx 1460-1475 → scaled to 1024
true_s = int(1460 * 1024 / len(vals))
true_e = int(1475 * 1024 / len(vals))

QUERIES = [
    'Find all periods with oscillations',
    'Detect stale data',
    'Identify temperature spikes',
    'Find sudden increases',
    'Find periods with abnormal dynamics',
]

print(f'Running multi-scale segmentation on {TAG}...')
max_prob, binary, metrics = multiscale_segmentation(
    vals_ds, TAG, 'Identify temperature spikes',
    mlp=mlp,
    encode_text_fn=encode_text,
    get_chronos_fn=get_chronos_embedding_cached,
    true_anom_start=true_s,
    true_anom_end=true_e,
    threshold=FINAL_THRESHOLD,
)

if metrics:
    print(f'F1={metrics["F1"]:.3f}  Precision={metrics["precision"]:.3f}  Recall={metrics["recall"]:.3f}')

In [ ]:
# Heatmap across all queries
plot_segmentation_heatmap(
    vals_ds, TAG, AREA, QUERIES,
    mlp=mlp,
    encode_text_fn=encode_text,
    get_chronos_fn=get_chronos_embedding_cached,
    threshold=FINAL_THRESHOLD,
    save_path=f'../../figures/heatmap_{TAG.replace("-","_")}.png',
)

In [ ]:
# Clean up
shutdown_server()
print('Chronos server stopped.')